# Fase 7 — síntesis y traspaso al modelado

Este notebook audita los archivos gold persistidos y resume las pruebas de fases 4 y 5. Cada CSV conserva una fila por **semana objetivo** para `h=2` o `h=4`. La etiqueta `brote` es un resultado observado de esa semana. Se excluyen del modelo `casos_Dengue`, el umbral de brote, las llaves y los metadatos. **Aquí no se entrena XGBoost ni se fija un corte de test.**

In [5]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from src.modeling.contrato_xgboost import NO_PREDICTORAS, auditar_traspaso, resumir_ablaciones

reporte = auditar_traspaso()
reporte["columnas_no_predictoras"] = list(NO_PREDICTORAS)
reporte["ablaciones_ridge_conteos"] = resumir_ablaciones()
salida = ROOT / "docs" / "feature_engineering" / "metricas" / "fase7_sintesis.json"
salida.parent.mkdir(parents=True, exist_ok=True)
salida.write_text(json.dumps(reporte, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print({"metricas": str(salida), "fuente": reporte["fuente"], "sha256_fuente": reporte["sha256_fuente"]})

{'metricas': 'c:\\Users\\Rosa\\RufoEsEternoElRegreso\\Denguekay\\docs\\feature_engineering\\metricas\\fase7_sintesis.json', 'fuente': 'meteo_socio_epi_piura_2017_2025.csv', 'sha256_fuente': 'b04dc8c31793a08b417e5bafd9db101f4a18c9299686cb2dd4bfe2e5049b6c88'}


## Contrato de variables

La base inicial sugerida para la futura comparación con XGBoost es **historia propia + calendario**. Vecinos, contexto provincial/regional, clima y demografía quedan como bloques separados de ablación. Que una columna exista en gold no garantiza que estuviera publicada en el origen histórico del pronóstico.

In [6]:
filas = []
for h, info in reporte["horizontes"].items():
    for familia, columnas in info["familias"].items():
        filas.append({"h": int(h), "familia": familia, "n_columnas": len(columnas),
                      "filas_con_algun_nulo": info["nulos_por_familia"][familia],
                      "columnas": ", ".join(columnas)})
display(pd.DataFrame(filas))
print("Exclusiones:", reporte["columnas_no_predictoras"])

,h,familia,n_columnas,filas_con_algun_nulo,columnas
0,2,historia,4,325,"casos_lag_2, casos_lag_3, casos_media_4_h2, ca..."
1,2,calendario,2,0,"semana_epi_seno, semana_epi_coseno"
2,2,vecinos,2,130,"vecinos_media_casos_knn5_h2, vecinos_frac_con_..."
3,2,jerarquia,4,130,"provincia_otros_casos_h2, provincia_otros_frac..."
4,2,clima,2,390,"hum_rel_media_media5_h2, precip_total_mm_media..."
5,2,socio_2017,4,0,"fraccion_rural_2017, fraccion_menores_15_2017,..."
6,2,socio_anual_reconstruida,16,0,"fraccion_rural_anual, fraccion_mujeres_anual, ..."
7,4,historia,4,455,"casos_lag_4, casos_lag_5, casos_media_4_h4, ca..."
8,4,calendario,2,0,"semana_epi_seno, semana_epi_coseno"
9,4,vecinos,2,260,"vecinos_media_casos_knn5_h4, vecinos_frac_con_..."


Exclusiones: ['casos_Dengue', 'brote', 'umbral_brote_casos', 'ubigeo', 'anio', 'semana', 'provincia', 'distrito', 'distrito_key', 'semana_inicio', 'origen_inicio', 'origen_cierre', 'socio_2017_disponible_al_origen', 'socio_fracciones_interpoladas_con_2025']


## Etiqueta y cobertura

Las filas iniciales de cada distrito carecen de suficiente historia para algunas ventanas. Permanecen con `NaN`; la selección de filas y el tratamiento de nulos deben decidirse dentro del entrenamiento de cada horizonte. Los 2025-S53 de Sala no están asignados a gold.

In [7]:
resumen = []
for h, info in reporte["horizontes"].items():
    resumen.append({"h": int(h), "filas": info["filas"], "distritos": info["distritos"],
                    "candidatas": info["candidatas"],
                    "filas_candidatas_completas": info["filas_candidatas_completas"],
                    "brote_positivos": info["positivos_brote"]})
display(pd.DataFrame(resumen))
print("Positivos por año:", reporte["horizontes"]["4"]["positivos_por_anio"])
print("Regla:", {k: reporte["regla_brote"][k] for k in ("multiplicador", "anios_previos", "minimo_casos")})

,h,filas,distritos,candidatas,filas_candidatas_completas,brote_positivos
0,2,30485,65,34,30095,3002
1,4,30485,65,34,29965,3002


Positivos por año: {'2017': 520, '2018': 5, '2019': 0, '2020': 2, '2021': 213, '2022': 354, '2023': 1402, '2024': 503, '2025': 3}
Regla: {'multiplicador': 1.5, 'anios_previos': 5, 'minimo_casos': 2}


## Evidencia exploratoria y límite

Las ablaciones de clima y demografía proceden de **ridge para conteos**, con temporadas 2022–2025 y cortes temporales. Sus deltas MAE no miden rendimiento de XGBoost ni precisión de la etiqueta `brote`. No se hizo una ablación predictiva del bloque de vecinos.

In [9]:
filas = []
for fase, modelos_por_h in reporte["ablaciones_ridge_conteos"].items():
    for h, modelos in modelos_por_h.items():
        for nombre, resultado in modelos.items():
            filas.append({"fase": fase, "h": int(h), "modelo": nombre,
                          "referencia": resultado["referencia"],
                          "cortes_con_menor_mae": resultado["cortes_con_menor_mae"],
                          "n_cortes": len(resultado["temporadas"])})
display(pd.DataFrame(filas))

,fase,h,modelo,referencia,cortes_con_menor_mae,n_cortes
0,clima,2,historia_calendario_clima,historia_calendario,2,4
1,clima,4,historia_calendario_clima,historia_calendario,3,4
2,sociodemografia,2,mas_15_fracciones_anuales,historia_calendario,1,4
3,sociodemografia,2,mas_clima,historia_calendario,2,4
4,sociodemografia,2,mas_clima_poblacion_y_eje,mas_clima,2,4
5,sociodemografia,2,mas_clima_poblacion_y_eje_anual,mas_clima,1,4
6,sociodemografia,2,mas_eje_urbano_2017,historia_calendario,3,4
7,sociodemografia,2,mas_eje_urbano_anual,historia_calendario,2,4
8,sociodemografia,2,mas_poblacion_2017,historia_calendario,3,4
9,sociodemografia,2,mas_poblacion_anual,historia_calendario,3,4


## Entrega al modelado

El contrato, las decisiones abiertas y la matriz de disponibilidad están en `docs/feature_engineering/fase7_sintesis_traspaso.md`. Para cada fold deben separarse filas por **tiempo compartido entre distritos**, exigir que las etiquetas de entrenamiento estuvieran observadas antes del primer origen de prueba y ajustar cualquier climatología, PCA, imputación o codificación únicamente con entrenamiento. La publicación real de casos, clima y proyecciones aún requiere verificación. La fase de modelado necesita aprobación específica.